# async def

## 核心组件拆解
### async def：定义一个“可以暂停的函数”
   - 作用：用 async def 定义的函数叫协程函数（Coroutine Function）。它和普通 def 函数的最大区别是：<font color='red'>调用它时，函数体内的代码不会立刻执行，而是返回一个协程对象（Coroutine Object）。</font>
   - 通俗理解：它就像一份“待办任务清单”，你只是把清单写好了，但还没开始照着做。
### await：交出控制权的“暂停键”
   - 作用：只能在 async def 内部使用。<font color='red'>当程序执行到 await 时，它会主动暂停当前任务，把控制权交还给大管家（事件循环），并去等待某个耗时操作（如网络请求、文件读写、asyncio.sleep）完成。</font>
   - 通俗理解：服务员对客人说“点好了叫我”，然后转身去忙别的。
### asyncio：幕后大管家（事件循环）
   - 作用：asyncio 是 Python 的标准库，它的核心是事件循环（Event Loop）。<font color='red'>它负责不断地检查哪些任务在等待、哪些任务已经准备好继续执行，并高效地在这些任务之间来回切换。</font>
   - 通俗理解：餐厅的调度中心，时刻盯着所有桌子的状态，指挥服务员在最佳时机去服务最佳的客人。

In [ ]:
import asyncio

# 1. 定义异步函数 (协程函数)
async def make_coffee(name):
    print(f"☕ 开始为 {name} 制作咖啡...")
    # 模拟耗时的 I/O 操作（比如等咖啡机萃取），这里必须用 asyncio.sleep
    await asyncio.sleep(2)  
    print(f"✅ {name} 的咖啡做好了！")
    return f"{name}的咖啡"

# 2. 主调度函数
async def main():
    # 创建多个并发任务（相当于来了三桌客人同时点单）
    # asyncio.create_task 会把协程包装成任务(Task)，交给事件循环并发执行
    task1 = asyncio.create_task(make_coffee("张三"))
    task2 = asyncio.create_task(make_coffee("李四"))
    task3 = asyncio.create_task(make_coffee("王五"))

    # 等待所有任务完成
    await asyncio.gather(task1, task2, task3)

# 3. 启动事件循环 (Python 3.7+ 的标准入口)
asyncio.run(main())

## 适用场景与局限性
- 最适合的场景（I/O 密集型）：
  - 程序的瓶颈在于“等待”。例如：高并发网络爬虫、Web 服务器（如 FastAPI）、微服务间通信、数据库查询、文件读写等。在这些场景下，asyncio 能用单线程实现极高的并发量。

- 不适合的场景（CPU 密集型）：
  - 程序的瓶颈在于“疯狂计算”。例如：复杂的数学运算、大规模图像处理、机器学习模型训练等。因为 asyncio 本质上是单线程的，遇到纯计算任务依然会阻塞，无法利用多核 CPU 的优势。对于这类任务，应该使用多进程（multiprocessing）。
  
## ⚠️ 5. 新手最容易踩的坑
- 忘记加 await：
  - 调用异步函数时如果没加 await（也没用 create_task），它只会返回一个协程对象，函数体内的代码压根不会执行。

- 在异步函数里用了同步阻塞操作：
  - 这是大忌！比如在 async def 里用了 time.sleep() 或同步的 requests.get()。这会直接把整个事件循环（整个餐厅）卡死，所有任务都得陪着一起等。必须使用异步版本的库（如 aiohttp 代替 requests，asyncio.sleep 代替 time.sleep）。

- 以为写了 async def 就自动并发：
  - <font color='red'>async def 只是定义了能并发的函数，想要真正并发，必须通过 asyncio.create_task() 或 asyncio.gather() 等方式把它们交给事件循环去调度。</font>

# asyncio

- asyncio 库提供了丰富的方法来管理异步编程的方方面面。
- 为了让你更清晰地掌握，我们可以把这些方法按照功能划分为五大类：
    - 程序入口与基础工具、
    - 任务创建与并发、
    - 超时与取消、
    - 并发控制，
    - 以及结构化并发（Python 3.11+ 的新特性）。
    
下面将详细介绍这些核心方法并附上实战示例：
## 🚀 1. 程序入口与基础工具
- asyncio.run(coro)：
  - 这是 Python 3.7+ 推荐的唯一程序入口。它会自动创建一个新的事件循环，运行传入的协程，并在结束后完美地关闭循环、清理资源。
- asyncio.sleep(delay)：
  - 绝对不能使用 time.sleep()！ asyncio.sleep() 是一个非阻塞的挂起操作。它会告诉事件循环：“我先休息 delay 秒，这期间你可以去调度其他任务。”
- asyncio.gather(*aws)：
  - 用于并发运行多个任务，并等待它们全部完成。它会按传入的顺序返回结果列表。


In [ ]:
import asyncio

async def fetch_data(id, delay):
    print(f"任务 {id} 开始请求...")
    await asyncio.sleep(delay)  # 模拟非阻塞的网络等待
    print(f"任务 {id} 请求完成！")
    return f"结果-{id}"

async def main():
    # 并发执行三个任务，总耗时取决于最慢的那个（约3秒），而不是总和
    results = await asyncio.gather(
        fetch_data(1, 2),
        fetch_data(2, 3),
        fetch_data(3, 1)
    )
    print(f"所有结果: {results}")

# 程序的唯一入口
asyncio.run(main())

## 🏃 2. 任务创建与并发调度
- asyncio.create_task(coro)：
  - 将一个协程包装成任务（Task）并立即交给事件循环并发执行。<font color='red'>这是实现“后台并发”最常用的方法。</font>
- asyncio.as_completed(aws)：
  - 与 gather 不同，它谁先完成就先返回谁。非常适合处理需要实时响应最快结果的场景。
  
并发调度示例：

In [1]:
import asyncio
import random

async def random_task(i):
    delay = random.uniform(0.5, 2.0)
    await asyncio.sleep(delay)
    return f"任务-{i} 耗时 {delay:.2f}秒"

async def main():
    # 1. 使用 create_task 提前创建任务，让它们立刻开始在后台跑
    task1 = asyncio.create_task(random_task(1))
    task2 = asyncio.create_task(random_task(2))
    
    # 做点别的事情...
    print("主程序在做其他准备工作...")
    
    # 2. 使用 as_completed 谁先跑完先处理谁的结果
    tasks = [random_task(i) for i in range(3, 6)]
    for coro in asyncio.as_completed(tasks):
        result = await coro
        print(f"【抢先拿到结果】: {result}")

    # 等待之前提前创建的任务完成
    await task1
    await task2

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

## ⏱️ 3. 超时与取消控制
- asyncio.wait_for(aw, timeout)：
  - 为某个协程或任务设置超时时间。如果在规定时间内没完成，会抛出 asyncio.TimeoutError。
- task.cancel()：
  - 手动取消一个正在运行的任务。被取消的任务会在最近的 await 挂起点抛出 asyncio.CancelledError。
  
超时与取消示例：

In [ ]:
import asyncio

async def slow_operation():
    try:
        print("开始一个很慢的操作...")
        await asyncio.sleep(5)
        return "操作成功完成"
    except asyncio.CancelledError:
        print("任务被外部取消了！正在清理资源...")
        raise

async def main():
    # 1. 超时控制：只等2秒，超过就报错
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=2)
        print(result)
    except asyncio.TimeoutError:
        print("操作超时了！")

    # 2. 手动取消任务
    task = asyncio.create_task(slow_operation())
    await asyncio.sleep(1) # 等1秒后反悔了，决定取消它
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("主程序确认任务已被取消。")

asyncio.run(main())

## 🚦 4. 并发控制（限流）
- asyncio.Semaphore(n)：
  - 信号量。用于限制同时运行的任务数量。比如你有一个 API 接口限制每秒最多 10 次请求，就可以用 Semaphore(10) 来防止把对方服务器打挂。
  
限流示例：

In [ ]:
import asyncio

async def limited_task(semaphore, task_id):
    # 只有拿到信号量（通行证）才能进入
    async with semaphore:
        print(f"任务 {task_id} 获得了通行证，开始执行！")
        await asyncio.sleep(1)
        print(f"任务 {task_id} 执行完毕，归还通行证。")

async def main():
    # 限制同时最多只有 3 个任务在执行
    semaphore = asyncio.Semaphore(3)
    tasks = [limited_task(semaphore, i) for i in range(10)]
    await asyncio.gather(*tasks)

asyncio.run(main())

## 🏗️ 5. 结构化并发（Python 3.11+ 强烈推荐）
- asyncio.TaskGroup()：
  - 这是 Python 3.11 引入的现代并发模式，旨在替代大部分 gather 的使用。它能确保组内的所有任务都能被妥善管理，不会出现“孤儿任务”。如果组内有一个任务失败，它会自动取消其他所有任务，并将异常打包成 ExceptionGroup 抛出，极大地提高了程序的健壮性。
- asyncio.timeout()：
  - 3.11+ 引入的上下文管理器，比 wait_for 写法更优雅。
  
TaskGroup 示例：

In [ ]:
import asyncio

async def task_a():
    await asyncio.sleep(1)
    print("任务 A 完成")
    # raise ValueError("任务 A 失败了！") # 如果取消这行注释，任务 B 会被自动取消

async def task_b():
    await asyncio.sleep(3)
    print("任务 B 完成")

async def main():
    # 使用 TaskGroup 管理并发任务
    async with asyncio.TaskGroup() as tg:
        task1 = tg.create_task(task_a())
        task2 = tg.create_task(task_b())
        print("TaskGroup 内的任务已提交，等待它们完成...")
    
    # 退出 async with 块时，保证所有任务要么成功，要么被取消
    print(f"任务 A 的结果: {task1.result()}")

asyncio.run(main())

## 总结建议：
- 在日常开发中，建议优先使用 asyncio.run() 作为入口，
- 用 create_task 和 TaskGroup（如果是 Python 3.11+）来管理并发，
- 用 Semaphore 做限流，
- 并时刻注意使用 await asyncio.sleep() 代替任何同步的阻塞操作。

## ⚠️ 补充说明：关于 loop.run_until_complete()
- <font color='red'>在 Python 3.10 之前</font>，很多人喜欢用 asyncio.get_event_loop().run_until_complete(coro) 来替代 asyncio.run()。
- 但<font color='red'>在 Python 3.10+ 中</font>，官方已经不再推荐这种写法，且在嵌套场景下它的行为更难预测，容易引发更隐蔽的 Bug。
- 因此，请始终坚持在程序最顶层入口使用 asyncio.run()，在内部使用 await 或 asyncio.create_task()。

### 为什么在Python 3.10+ 中，官方已经不再推荐这种写法？

在 Python 3.10+ 中，官方不再推荐使用 asyncio.get_event_loop() 和 loop.run_until_complete() 这种传统写法，
- <font color='red'>核心原因是为了推动异步编程从“<b>隐式魔法</b>”走向“<b>显式明确</b>”，从而解决旧 API 带来的行为模糊和难以排查的 Bug。</font>

###### 具体来说，主要有以下几个维度的考量：
1. 拒绝“隐式自动创建”的魔法行为
- <font color='red'>在 Python 3.10 之前，asyncio.get_event_loop() 有一个非常“智能”但也极其危险的特性：如果当前线程没有事件循环，它会在后台自动隐式地为你创建一个新的循环。</font>
- <font color='green'>带来的问题：这种隐式行为让开发者很难分清自己到底是在使用一个已经存在的循环，还是无意中创建了一个全新的循环。在复杂的项目中，这极易导致不同模块间创建了多个互不兼容的事件循环，引发严重的并发 Bug。</font>
- 3.10+ 的改变：从 3.10 开始，官方收紧了这一行为。如果当前没有运行的循环，get_event_loop() 不再自动创建，而是直接抛出 DeprecationWarning（弃用警告）或 RuntimeError。<font color='red'>官方希望你明确地知道自己到底是要“新建”还是“获取”。</font>

2. 无法保证获取到的循环“正在运行”
- get_event_loop() 只能保证返回一个循环对象，但不能保证这个循环当前是“活着”（正在运行）的。
- 带来的问题：<font color='red'>你可能拿到一个已经关闭、或者还没启动的循环对象，然后尝试在上面调度任务，结果导致程序卡死或报错。这种状态的不确定性增加了代码的脆弱性。</font>
- 替代方案：官方推荐使用 asyncio.get_running_loop()。顾名思义<font color='red'>，它只会在有循环真正跑起来的时候才返回对象，否则直接报错。这种“非黑即白”的明确反馈，能有效防止你在错误的时机去操作事件循环。</font>

3. 提倡“<font color='red'>显式优于隐式</font>”的设计哲学
- Python 之禅中有一条核心原则：“显式优于隐式（Explicit is better than implicit）”。
- 旧写法的模糊：<font color='red'>get_event_loop().run_until_complete() 这种写法把循环的创建、获取、运行混杂在一起，意图不够清晰。</font>
- 新写法的明确：
    - <font color='green'>如果你想在程序入口启动异步代码，官方提供了高度封装的 asyncio.run()。它会自动帮你完成“创建新循环 -> 运行协程 -> 完美关闭循环并清理资源”的全套动作，安全且省心。</font>
    - 如果你想在协程内部获取当前循环，请使用 asyncio.get_running_loop()，意图极其明确。
    - <font color='red'>如果你确实需要手动管理（比如在多线程等极端场景），官方希望你显式地调用 asyncio.new_event_loop() 来新建，而不是依赖 get_event_loop() 的隐式魔法。</font>
    
    
#### 💡 总结与建议
在 Python 3.10+ 的现代异步编程中，建议你遵循以下最佳实践：
- 程序入口：无脑使用 asyncio.run(main())，让它帮你托管整个生命周期。
- 协程内部：需要循环对象时，使用 asyncio.get_running_loop()。
- 彻底避坑：忘掉 asyncio.get_event_loop() 和 loop.run_until_complete()，<font color='red'>除非你在维护非常古老的遗留代码。</font>